# Backbone bake-off on EuroSAT: ResNet-50 vs ViT-S/16 vs ConvNeXt-T

**Question:** under one fixed training recipe, which backbone + init gives the best accuracy / speed trade-off on Sentinel-2 scene classification, and how does that change with label scarcity?

## Grid — `configs/grid.yaml`
| axis | values |
|---|---|
| backbone | `resnet50` (25.6M), `vit_small_patch16_224` (22.1M), `convnext_tiny` (28.6M) |
| init | **imagenet_rgb**: timm ImageNet weights, input = RGB (B04,B03,B02), `in_chans=3` — all backbones<br>**s2_ssl**: torchgeo SSL4EO-S12 MoCo weights, input = all 13 L1C bands, `in_chans=13` — ResNet-50, ViT-S<br>**imagenet_13ch**: ImageNet weights, stem stretched to 13ch (see *Models* section) — ConvNeXt-T (its 13-band row; addable to others to separate "more bands" from "in-domain pretraining") |
| mode | **lp** = linear probe (frozen backbone, cached features) · **ft** = full fine-tune |
| label fraction | 1% (≈162 imgs, ~16/class), 10% (1 620), 100% (16 200) of the train split; stratified, resampled per seed |
| seed | 0, 1, 2 (seed controls subset, head init, data order, aug) |

Weights:
- `resnet50.a1_in1k` · `vit_small_patch16_224.augreg_in21k_ft_in1k` · `convnext_tiny.fb_in22k`
- `ResNet50_Weights.SENTINEL2_ALL_MOCO` · `ViTSmall16_Weights.SENTINEL2_ALL_MOCO` (DINO also available, switch in config)
- **no S2-SSL ConvNeXt in torchgeo** → ConvNeXt's 13-band row uses `imagenet_13ch` instead.

Runs: FT = 6 (backbone,init) × 3 fractions × 3 seeds = **54**; LP = same 54 but cheap (features extracted once per backbone/init).

## Data — EuroSAT (torchgeo, 13-band L1C, 64×64, 10 classes)
- Official torchgeo split: train 16 200 / val 5 400 / test 5 400. Test only for final numbers.
- Whole dataset cached once as `int16` numpy (≈2.9 GB RAM). Batches indexed on CPU, preprocessing on GPU → no dataloader bottleneck, identical pipeline for every model.
- **Band order gotcha:** torchgeo EuroSAT stores `B8A` *last*; SSL4EO weights expect `…B08, B8A, B09…`. Cache is reordered to SSL4EO order.
- Upsampled 64→224 (bilinear) for all models: matches pretraining res, ViT pos-embeds, and makes throughput comparable. (torchgeo's own transform is resize 256 + center-crop 224; plain resize keeps the full tile.)

Normalisation (per init, matches how each was pretrained):
- `s2_ssl`: DN / 10 000 (exactly torchgeo's SSL4EO transform).
- `imagenet_rgb`: clip(DN / 3000, 0, 1) → model's own `pretrained_cfg` mean/std (**note** ViT augreg uses 0.5/0.5, not ImageNet stats).
- `imagenet_13ch`: per-band z-score with train stats (needed so the stretched stem's 3/13 scaling keeps activations at ImageNet scale).

## Recipe (identical for all backbones) — `configs/default.yaml`
**Full FT:** AdamW, lr 2e-4, wd 0.05 (no wd on norm/bias), 5% linear warmup → cosine, label smoothing 0.1, grad-clip 1.0, bs 64, bf16 autocast, channels_last. Aug = random D4 (h-flip, v-flip, transpose = all 8 dihedral variants; orientation-invariant for nadir EO). Epochs per fraction (same for every backbone): 1% → 200, 10% → 50, 100% → 15 (≈400 / 1 250 / 3 800 steps; same epochs across fractions would give 1% only a few dozen steps). Report **last-epoch** test top-1 (no early-stopping on test).

**Linear probe:** frozen backbone in eval, pooled pre-logit features (no aug), standardised; linear head trained with AdamW (100 epochs, cosine), wd chosen from a small grid on **val** → test top-1. Standard SSL-eval protocol.

## Efficiency metrics — `python -m bakeoff.bench`
Synthetic batch, bs 64, 224², bf16 autocast, channels_last, warmup then timed with `cuda.synchronize`. Reports:
- inference img/s + peak memory (`max_memory_allocated`), and
- train-step img/s + peak memory (fwd+bwd+AdamW) — relevant for FT vs LP cost.

Measured for `in_chans=3` and `13` (only stem changes; difference should be tiny). Weights irrelevant to speed → random init.

## Code layout
This notebook only **launches** and **analyses**; all training code is the `bakeoff` package (tested, `pytest`):
| module | role | CLI |
|---|---|---|
| `data` | EuroSAT int16 cache, band order, stratified subsets, GPU prep/aug | |
| `models` | `build(backbone, init)` for the 3 inits (see *Models* below) | |
| `config` | `RunConfig` from YAML + `--set k=v`; `config_hash` = run id (recipe only, no paths) | |
| `train` | one run: FT (Lightning) or LP (feature cache + linear head) → row in `results.csv` | `python -m bakeoff.train --config configs/default.yaml --set mode=lp` |
| `grid` | grid YAML → one subprocess per run; skips finished hashes; failures → `failures.csv` | `python -m bakeoff.grid configs/grid.yaml` |
| `bench` | throughput / peak mem per (backbone, in_chans) → `bench.csv` | `python -m bakeoff.bench --out bench.csv` |
| `analysis` | varied hparams from row configs, seed-mean summary, paired deltas, Pareto | |

Each `results.csv` row stores its full recipe (`config` JSON). Analysis groups by any recipe key that varies within a (mode, fraction) → sweeps show up as extra table columns / Pareto rows with no notebook edits. To pool recipes instead (e.g. lr tuned per backbone), pass `var=[...]` explicitly.

## Deliverables
1. `results.csv` (one row per run, resumable by `config_hash`) + `bench.csv` + `logs/<hash>.log`, `failures.csv`
2. Summary table: top-1 mean ± std over seeds, per (backbone, init, mode, fraction, varied hparams), joined with img/s & peak mem
3. FT−LP gap and seed-paired deltas (init vs init, hparam value vs value)
4. Pareto plot: accuracy vs inference throughput; cols = label fraction, rows = varied-hparam combo, frontier over FT points

## Caveats
- **Pretraining mismatch across ImageNet inits:** not same data/recipe → arch effect confounded with pretraining.
  | backbone | pretrain data | labels | recipe |
  |---|---|---|---|
  | ResNet-50 `a1_in1k` | IN-1k (1.3M) | supervised | RSB A1, 600 ep |
  | ViT-S `augreg_in21k_ft_in1k` | IN-21k → IN-1k ft | supervised | AugReg |
  | ConvNeXt-T `fb_in22k` | IN-22k (14M) | supervised | ConvNeXt, 90 ep |
  | ResNet-50 / ViT-S `s2_ssl` | SSL4EO-S12 (~1M S2 patches, 13 bands) | self-sup (MoCo) | SSL4EO |

  ViT-S/ConvNeXt-T see ~10× more pretraining data than ResNet-50 → part of any ResNet gap may be data, not arch. Kept as-is: these are the standard/recommended checkpoints. Parity alternative: all IN-1k (`convnext_tiny.fb_in1k`, `vit_small_patch16_224.augreg_in1k`).
- One shared recipe ≠ each backbone's best recipe (ViTs usually want layer-decay / stronger aug).
- LP uses the full val split for wd selection even at 1% labels — slightly optimistic, applied equally to all.
- Throughput is hardware-specific; compare only within one run of this notebook.
- `cudnn.benchmark=True` for speed → runs are not bit-exact reproducible; seeds still fix subsets/init.

Set `QUICK = True` first for a smoke test (`configs/quick.yaml`: 1 seed, 10%, 2 FT epochs; separate `quick/` out dir).

In [ ]:
import os, sys, site, importlib, importlib.util, subprocess

# deps missing in jupyter pod / Colab (runs are subprocesses of this kernel's python -> same env)
try:
    import timm, torchgeo, lightning, yaml, matplotlib, tabulate
except ModuleNotFoundError:
    %pip install --quiet "timm>=1.0" "torchgeo>=0.7" "lightning>=2.4" pyyaml matplotlib tabulate
    site.addsitedir(site.getusersitepackages())  # k8s pod: user site may not exist at kernel start
    importlib.invalidate_caches()

# bakeoff package: notebook dir if present, else shallow-clone repo to WORK_DIR (pod/Colab)
REPO, BRANCH = "https://github.com/ignaciomasari/cv_playground", "main"
WORK_DIR = "/home/jovyan/work" if os.path.isdir("/home/jovyan/work") else ".."  # PVC in pod
if importlib.util.find_spec("bakeoff") is None:
    src = os.path.join(WORK_DIR, "cv_playground")
    if not os.path.isdir(src):
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, src], check=True)
    else:
        subprocess.run(["git", "-C", src, "pull", "--ff-only"], check=True)
    sys.path.insert(0, os.path.join(src, "backbone_bakeoff"))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from bakeoff import analysis as an
from bakeoff.grid import PKG_ROOT, child_env, expand, launch
from bakeoff.results import read_results
print("bakeoff:", PKG_ROOT)


## Config

In [ ]:
QUICK = False  # True: configs/quick.yaml smoke test (own out dir)
GRID = os.path.join(PKG_ROOT, "configs", "quick.yaml" if QUICK else "grid.yaml")
TIMEOUT_H = 6  # per run; hung run killed -> failures.csv, grid continues

runs = expand(GRID)  # bad keys / values raise here
OUT_DIR = runs[0].cfg.out_dir
RESULTS_CSV, BENCH_CSV = runs[0].cfg.results_csv, os.path.join(OUT_DIR, "bench.csv")
print(f"{len(runs)} runs | results -> {os.path.abspath(RESULTS_CSV)}")


## Throughput benchmark
bs 64, 224², bf16, channels_last, random weights; `in_chans` 3 and 13. Inference (`inference_mode`) and train step (fwd + bwd + AdamW).

In [ ]:
# subprocess -> GPU memory freed after; delete bench.csv to re-measure
if not os.path.isfile(BENCH_CSV):
    os.makedirs(OUT_DIR, exist_ok=True)
    p = subprocess.run([sys.executable, "-m", "bakeoff.bench", "--out", BENCH_CSV],
                       env=child_env(), capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-3000:] if p.returncode else "")
    p.check_returncode()
pd.read_csv(BENCH_CSV).round(1)


## Run the grid (resumable)
One subprocess per run (`python -m bakeoff.train`), output in `OUT_DIR/logs/<hash>.log`. Finished runs (hash in `results.csv`) are skipped → re-run the cell after an interruption; failed runs are retried.
Changing any recipe value in the YAMLs → new hash → those runs run again (old rows stay, analysis separates them by the varied key).

Long grids: a dropped browser / dead kernel kills the launcher. From a pod terminal instead (survives disconnects):
```bash
cd <repo>/backbone_bakeoff && nohup python -m bakeoff.grid configs/grid.yaml --timeout-h 6 > grid.log 2>&1 &
```
Rows written by the old notebook loop have no `config_hash` → not counted as done.

In [ ]:
launch(GRID, dry_run=True);  # list pending


In [ ]:
out = launch(GRID, timeout_s=TIMEOUT_H * 3600)
fails = os.path.join(OUT_DIR, "failures.csv")
if out["failed"]:
    display(pd.read_csv(fails).tail(out["failed"]))
out


## Results table — test top-1 mean ± std over seeds + efficiency

In [ ]:
res = read_results(RESULTS_CSV)
assert res is not None, f"no results at {RESULTS_CSV} (or old schema -> .bak)"
if res.config_hash.notna().any():  # old notebook rows (no hash/config) would double-count seeds
    res = res[res.config_hash.notna()]
bench = pd.read_csv(BENCH_CSV) if os.path.isfile(BENCH_CSV) else None
VAR = an.varied(res)  # recipe keys swept within a (mode, frac), e.g. ["lr"]
agg = an.summary(res, bench)
agg.to_csv(os.path.join(OUT_DIR, "summary_long.csv"), index=False)

table = an.table(agg, VAR)
table.to_csv(os.path.join(OUT_DIR, "summary_table.csv"))
with open(os.path.join(OUT_DIR, "summary_table.md"), "w") as f:
    f.write(table.to_markdown())
print(f"{len(res)} runs | varied: {VAR or 'none'} | test top-1 % mean ± std over seeds | bench: bs 64, 224px, bf16")
table


## FT − LP gap and paired deltas (pp, mean ± std over seeds)
Paired = same fraction + seed (→ same label subset) and same recipe; only the compared key differs. Std of a paired delta is much smaller than the std of either arm.

In [ ]:
display(an.ft_lp_gap(res).round(2).rename(columns=lambda c: f"FT-LP {c}" if c in ("mean", "std") else c))

# init contrasts: s2_ssl−imagenet_rgb = bands + in-domain; 13ch−rgb = bands; s2_ssl−13ch = in-domain pretraining
for a, b in [("s2_ssl", "imagenet_rgb"), ("imagenet_13ch", "imagenet_rgb"), ("s2_ssl", "imagenet_13ch")]:
    d = an.paired_delta(res, "init", a, b)
    if len(d):
        print(f"{a} − {b}"); display(d.round(2))

# swept hparams: each value vs the first
res_v, _ = an.with_varied(res)
for k in VAR:
    vals = sorted(res_v[k].dropna().unique(), key=str)
    for v in vals[1:]:
        print(f"{k}={v} − {k}={vals[0]}"); display(an.paired_delta(res, k, v, vals[0]).round(2))


## Pareto plot — accuracy vs inference throughput
Cols = label fraction, rows = varied-hparam combo (one row if nothing swept). Colour = backbone, marker = init, filled = FT, hollow = LP, error bars = std over seeds. Dashed = Pareto frontier over FT points (no other point both faster and more accurate).

In [ ]:
if bench is None:
    print("no bench.csv -> run the benchmark cell")
else:
    fig = an.plot_pareto(agg, VAR, xlabel="inference img/s (bf16, bs 64)")
    fig.savefig(os.path.join(OUT_DIR, "pareto.png"), dpi=150, bbox_inches="tight")
    plt.show()


## Reading the results (fill in after running)
Things to check / expected patterns to confirm or refute:
- **Label efficiency:** S2-SSL init should help most at 1% and in **LP** (in-domain features, all 13 bands); gap shrinks at 100% FT.
- **LP vs FT:** ImageNet-RGB features usually have a large FT−LP gap on EuroSAT; SSL4EO features a smaller one.
- **Bands vs pretraining:** if `imagenet_13ch` is enabled, `imagenet_13ch − imagenet_rgb` isolates the value of extra bands; `s2_ssl − imagenet_13ch` isolates in-domain pretraining.
- **Speed:** at 224² ResNet-50 and ConvNeXt-T are typically close; ViT-S/16 (196 tokens) is often the fastest in bf16 on recent GPUs. Stem `in_chans` 3→13 should barely move img/s.
- **Pareto:** state which configs sit on the frontier per label regime, and the recommendation for a deployment that needs X img/s.

## Models + per-init input spec (`bakeoff/models.py`)
`build(bb, init, num_classes)` → `(model, spec)`. `spec` = bands, scale, clip, mean, std used by `prep` on GPU.
S2-SSL weights loaded into a plain timm model (same as `torchgeo.models.resnet50(weights=...)` does internally), with a check that only head/final-norm keys are missing.

### `imagenet_13ch`: stretching an RGB stem to 13 bands
`timm.create_model(..., pretrained=True, in_chans=13)` loads all ImageNet weights unchanged except the first conv (ConvNeXt `stem.0`: 4×4 stride-4, weight `[96, 3, 4, 4]` → `[96, 13, 4, 4]`), adapted by `timm.models.adapt_input_conv`:
```python
w = w.repeat(1, ceil(13 / 3), 1, 1)[:, :13]  # tile R,G,B filter slices cyclically
w *= 3 / 13                                   # keep output magnitude
```
Band *i* (SSL order) gets the filter slice of RGB channel *i mod 3*:

| band | B01 | B02 | B03 | B04 | B05 | B06 | B07 | B08 | B8A | B09 | B10 | B11 | B12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| filter | R | G | B | R | G | B | R | G | B | R | G | B | R |

- Why 3/13: a stem output is Σ over input channels. With inputs of similar stats (per-band z-score), 13 terms instead of 3 would blow up activations ~13/3×; the rescale keeps the first feature map at ImageNet scale so the pretrained norms/blocks downstream still see familiar statistics.
- Effect: at step 0 each output ≈ the RGB filter response averaged over "pseudo-R/G/B" band groups — not the true RGB image (B02/B03 get G/B, not B/G). Positional, not spectral, mapping. FT learns to re-weight bands; LP is stuck with this mix.
- Rest of the network (stages, norms) = ImageNet-22k weights; head = new random 10-class layer.